<h1 align=center> Pandas for AI </h1>


**What is Pandas?** A Python library for working with **tables of data** — rows and columns, like an Excel sheet.

**Why learn it for AI?** Before a model can learn, the data must be **loaded, checked, cleaned and turned into numbers**. We study Pandas because it is the leading Python library for fast, efficient data analysis, cleaning, and manipulation of structured tabular data.

**Pandas and NumPy:** Pandas is built on NumPy. Every column is a NumPy array underneath. Pandas adds **names** for rows and columns, **different data types** per column, and tools for **missing values**.

**Before you start:** you should know basic Python (variables, lists, dictionaries, functions). NumPy basics help.

**Difficulty levels** — every section is tagged:

| Tag | Level | Meaning |
|---|---|---|
| 🟢 | Beginner | start here — the everyday basics |
| 🟡 | Intermediate | once the basics feel easy |
| 🔴 | Advanced | real ML data preparation |

**Roadmap**

| 🟢 Part 1 — Beginner | 🟡 Part 2 — Intermediate | 🔴 Part 3 — Advanced |
|---|---|---|
| 0. Setup | 9. Data Types | 20. Views, Copies & Copy-on-Write |
| 1. Series & DataFrame | 10. Text (String) Operations | 21. Method Chaining |
| 2. Looking at Your Data | 11. Applying Functions | 22. Speed & Memory |
| 3. Selecting Data | 12. Outliers & Binning | 23. Common Mistakes |
| 4. Adding, Renaming, Dropping | 13. Encoding Categories | 24. Pandas for AI |
| 5. Sorting | 14. Combining Tables | Cheat Sheet |
| 6. Missing Data | 15. GroupBy | Checklist |
| 7. Duplicates | 16. Reshaping | Practice + Solutions |
| 8. Basic Statistics | 17. Dates & Time Series | |
| ✅ Part 1 Recap | 18. Plotting · 19. Reading & Saving Files | |

Run each cell in order with **Shift + Enter**.

---
# 🟢 PART 1 — BEGINNER

## 0. Setup 🟢

We import Pandas as `pd` and NumPy as `np`. Everyone uses these short names.

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)        # makes random numbers the same every time you run
print("pandas version:", pd.__version__)

pandas version: 3.0.6


## 1. Series & DataFrame 🟢

Pandas has two main objects:

| Object | Shape | In simple words |
|---|---|---|
| **Series** | 1D | **one column** with labels |
| **DataFrame** | 2D | **a whole table** — many Series side by side |

The row labels are called the **index**. By default they are `0, 1, 2, ...`.

### 1.1 Series

In [3]:
ages = pd.Series([20, 22, 19, 21])
print(ages)

0    20
1    22
2    19
3    21
dtype: int64


The left column is the **index**, the right column is the **values**. `dtype` is the data type.

You can give your own labels, or build a Series from a dictionary (keys become the index):

In [4]:
scores = pd.Series({"Ali": 85, "Sara": 92, "Omar": 78})
print(scores)
print(scores["Sara"])       # get a value by its label
print(scores + 5)           # math works on every value at once

Ali     85
Sara    92
Omar    78
dtype: int64
92
Ali     90
Sara    97
Omar    83
dtype: int64


**Watch out:** a missing value is written `np.nan` (Not a Number). Because `NaN` is a decimal number, a column of integers with one `NaN` becomes **float**:

In [5]:
s = pd.Series([1, 3, 5, np.nan, 6, 8])
print(s)

0    1.0
1    3.0
2    5.0
3    NaN
4    6.0
5    8.0
dtype: float64


**Labels matter:** when you add two Series, Pandas matches values **by label**, not by position. A label that exists in only one of them gives `NaN`.

In [6]:
bonus = pd.Series({"Sara": 3, "Omar": 5, "Zahra": 10})
print(scores + bonus)                      # Ali and Zahra -> NaN
print(scores.add(bonus, fill_value=0))     # treat a missing label as 0

Ali       NaN
Omar     83.0
Sara     95.0
Zahra     NaN
dtype: float64
Ali      85.0
Omar     83.0
Sara     95.0
Zahra    10.0
dtype: float64


### 1.2 DataFrame

The most common way to make a DataFrame: a **dictionary** where each key is a column name and each value is a list.

In [7]:
people = pd.DataFrame({
    "name": ["Ali", "Sara", "Omar"],
    "age": [25, 30, 22],
    "city": ["Kabul", "Herat", "Mazar"]
})
print(people)

   name  age   city
0   Ali   25  Kabul
1  Sara   30  Herat
2  Omar   22  Mazar


Other ways you'll see:

In [8]:
# A list of dictionaries (one per row) - this is how data from APIs / JSON usually looks
records = pd.DataFrame([
    {"name": "Ali", "age": 25},
    {"name": "Sara", "age": 30, "city": "Herat"}     # missing keys become NaN
])
print(records)

# From a NumPy array, giving column names
grid = pd.DataFrame(np.arange(6).reshape(3, 2), columns=["x", "y"])
print(grid)

# With dates as the index
dates = pd.date_range("2023-01-01", periods=4)
daily = pd.DataFrame(np.random.randn(4, 3), index=dates, columns=["A", "B", "C"])
print(daily)

   name  age   city
0   Ali   25    NaN
1  Sara   30  Herat
   x  y
0  0  1
1  2  3
2  4  5
                   A         B         C
2023-01-01  0.496714 -0.138264  0.647689
2023-01-02  1.523030 -0.234153 -0.234137
2023-01-03  1.579213  0.767435 -0.469474
2023-01-04  0.542560 -0.463418 -0.465730


### 1.3 Useful DataFrame properties

| Code | Gives you |
|---|---|
| `df.shape` | (number of rows, number of columns) |
| `df.columns` | the column names |
| `df.index` | the row labels |
| `df.dtypes` | the type of each column |
| `df.size` | total number of cells |
| `df.T` | the table flipped (rows ↔ columns) |
| `df.to_numpy()` | the data as a NumPy array — **what ML models take as input** |

print(people.shape)
print(people.columns.tolist())
print(people.index)
print(people.dtypes)
print(people.T)
print(grid.to_numpy())

## 2. Looking at Your Data 🟢

The **first thing** you do with any dataset is look at it: How big is it? What columns? Anything missing? Anything strange?

### 2.1 Our practice dataset

We'll use one small **student table** for most of the notebook. It has **real-world problems on purpose**, so we can learn to fix them:

- missing scores (`NaN`)
- messy text: `" kabul "` and `"HERAT"`
- a duplicate row
- an impossible age (99)
- dates stored as text

We also save it as a CSV file so we can practise loading files later.

In [9]:
students = pd.DataFrame({
    "student_id":  [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 103, 112],
    "name":        ["Ali", "Sara", "Omar", "Zahra", "Ahmad", "Fatima", "Hamid", "Maryam", "Yusuf", "Laila", "Omar", "Nadia"],
    "gender":      ["M", "F", "M", "F", "M", "F", "M", "F", "M", "F", "M", "F"],
    "province":    ["Kabul", "Herat", " kabul ", "Balkh", "Kabul", "HERAT", "Balkh", None, "Kabul", "Herat", " kabul ", "Kandahar"],
    "age":         [20, 22, 19, 21, 23, 20, 99, 22, 21, 19, 19, 24],
    "study_hours": [5, 8, 3, 6, 2, 9, 4, 7, 1, 6, 3, 5],
    "math":        [78, 92, 65, np.nan, 55, 95, 70, 88, 45, np.nan, 65, 72],
    "science":     [74, 89, 60, 80, 58, 91, 66, 85, 50, 77, 60, 70],
    "english":     [80, 85, 70, 90, 60, 88, 75, 82, 55, 79, 70, 68],
    "enroll_date": ["2023-01-15", "2023-02-03", "2023-01-20", "2023-03-11", "2023-02-27", "2023-01-05",
                    "2023-04-18", "2023-03-02", "2023-05-09", "2023-02-14", "2023-01-20", "2023-04-01"],
    "passed":      ["yes", "yes", "yes", "yes", "no", "yes", "yes", "yes", "no", "yes", "yes", "yes"],
})
students.to_csv("students.csv", index=False)
students

,student_id,name,gender,province,age,study_hours,math,science,english,enroll_date,passed
0,101,Ali,M,Kabul,20,5,78.0,74,80,2023-01-15,yes
1,102,Sara,F,Herat,22,8,92.0,89,85,2023-02-03,yes
2,103,Omar,M,kabul,19,3,65.0,60,70,2023-01-20,yes
3,104,Zahra,F,Balkh,21,6,NaN,80,90,2023-03-11,yes
4,105,Ahmad,M,Kabul,23,2,55.0,58,60,2023-02-27,no
5,106,Fatima,F,HERAT,20,9,95.0,91,88,2023-01-05,yes
6,107,Hamid,M,Balkh,99,4,70.0,66,75,2023-04-18,yes
7,108,Maryam,F,NaN,22,7,88.0,85,82,2023-03-02,yes
8,109,Yusuf,M,Kabul,21,1,45.0,50,55,2023-05-09,no
9,110,Laila,F,Herat,19,6,NaN,77,79,2023-02-14,yes


> **Tip:** writing a variable name as the **last line** of a cell displays it as a nice table. `print()` shows plain text.

### 2.2 `head()`, `tail()`, `sample()`

| Method | Shows |
|---|---|
| `head(n)` | first *n* rows (default 5) |
| `tail(n)` | last *n* rows |
| `sample(n)` | *n* random rows — good for spotting problems |

In [10]:
print(students.head(3))
print(students.tail(2))
print(students.sample(2, random_state=1))

   student_id  name gender province  age  study_hours  math  science  english  \
0         101   Ali      M    Kabul   20            5  78.0       74       80   
1         102  Sara      F    Herat   22            8  92.0       89       85   
2         103  Omar      M   kabul    19            3  65.0       60       70   

  enroll_date passed  
0  2023-01-15    yes  
1  2023-02-03    yes  
2  2023-01-20    yes  
    student_id   name gender  province  age  study_hours  math  science  \
10         103   Omar      M    kabul    19            3  65.0       60   
11         112  Nadia      F  Kandahar   24            5  72.0       70   

    english enroll_date passed  
10       70  2023-01-20    yes  
11       68  2023-04-01    yes  
   student_id   name gender province  age  study_hours  math  science  \
2         103   Omar      M   kabul    19            3  65.0       60   
3         104  Zahra      F    Balkh   21            6   NaN       80   

   english enroll_date passed  
2     

### 2.3 `shape` and `info()`

`info()` is the most useful first check. It shows every column, how many values are **not missing**, and the type.

In [11]:
print(students.shape)
students.info()

(12, 11)
<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   12 non-null     int64  
 1   name         12 non-null     str    
 2   gender       12 non-null     str    
 3   province     11 non-null     str    
 4   age          12 non-null     int64  
 5   study_hours  12 non-null     int64  
 6   math         10 non-null     float64
 7   science      12 non-null     int64  
 8   english      12 non-null     int64  
 9   enroll_date  12 non-null     str    
 10  passed       12 non-null     str    
dtypes: float64(1), int64(5), str(5)
memory usage: 1.2 KB


**Reading it:** there are 12 rows, but `math` has only **10** non-null values → 2 are missing. `province` has 11 → 1 missing. And `enroll_date` is text, not a date — we'll fix that in Section 9.

### 2.4 `describe()` — quick statistics

For number columns it shows count, mean, standard deviation, min, max and the quartiles (25%, 50% = median, 75%).

Look at `age`: the **max is 99** — `describe()` just found a mistake in our data.

In [12]:
students.describe()

,student_id,age,study_hours,math,science,english
count,12.000000,12.000000,12.000000,10.000000,12.000000,12.000000
mean,105.833333,27.416667,4.916667,72.500000,71.666667,75.166667
std,3.433348,22.601120,2.429303,16.119347,13.206977,10.903155
min,101.000000,19.000000,1.000000,45.000000,50.000000,55.000000
25%,103.000000,19.750000,3.000000,65.000000,60.000000,69.500000
50%,105.500000,21.000000,5.000000,71.000000,72.000000,77.000000
75%,108.250000,22.250000,6.250000,85.500000,81.250000,82.750000
max,112.000000,99.000000,9.000000,95.000000,91.000000,90.000000


In [13]:
students.describe(include="all")     # also summarises text columns: unique values, most common (top)

,student_id,name,gender,province,age,study_hours,math,science,english,enroll_date,passed
count,12.000000,12,12,11,12.000000,12.000000,10.000000,12.000000,12.000000,12,12
unique,NaN,11,2,6,NaN,NaN,NaN,NaN,NaN,11,2
top,NaN,Omar,M,Kabul,NaN,NaN,NaN,NaN,NaN,2023-01-20,yes
freq,NaN,2,6,3,NaN,NaN,NaN,NaN,NaN,2,10
mean,105.833333,NaN,NaN,NaN,27.416667,4.916667,72.500000,71.666667,75.166667,NaN,NaN
std,3.433348,NaN,NaN,NaN,22.601120,2.429303,16.119347,13.206977,10.903155,NaN,NaN
min,101.000000,NaN,NaN,NaN,19.000000,1.000000,45.000000,50.000000,55.000000,NaN,NaN
25%,103.000000,NaN,NaN,NaN,19.750000,3.000000,65.000000,60.000000,69.500000,NaN,NaN
50%,105.500000,NaN,NaN,NaN,21.000000,5.000000,71.000000,72.000000,77.000000,NaN,NaN
75%,108.250000,NaN,NaN,NaN,22.250000,6.250000,85.500000,81.250000,82.750000,NaN,NaN


### 2.5 Counting values

| Method | Gives |
|---|---|
| `nunique()` | how many different values each column has |
| `unique()` | the different values themselves |
| `value_counts()` | how often each value appears |

`value_counts()` shows the messy text clearly: `" kabul "` and `"HERAT"` are counted separately from `"Kabul"` and `"Herat"`.

In [14]:
print(students.nunique())
print(students["gender"].unique())
print(students["province"].value_counts(dropna=False))     # dropna=False also counts missing

student_id     11
name           11
gender          2
province        6
age             7
study_hours     9
math            9
science        11
english        11
enroll_date    11
passed          2
dtype: int64
<StringArray>
['M', 'F']
Length: 2, dtype: str
province
Kabul       3
Herat       2
 kabul      2
Balkh       2
HERAT       1
NaN         1
Kandahar    1
Name: count, dtype: int64


## 3. Selecting Data 🟢

| You want | Code | Returns |
|---|---|---|
| one column | `df["col"]` | Series |
| several columns | `df[["col1", "col2"]]` | DataFrame (note the **double** brackets) |
| rows/columns by **name** | `df.loc[rows, cols]` | end of a range is **included** |
| rows/columns by **position number** | `df.iloc[rows, cols]` | end of a range is **excluded** (like Python) |
| one single value | `df.at[row, col]` / `df.iat[i, j]` | a value |
| rows matching a condition | `df[condition]` or `df.query("...")` | DataFrame |

### 3.1 Columns

In [15]:
print(students["name"].head(3))
print(students[["name", "math"]].head(3))

0     Ali
1    Sara
2    Omar
Name: name, dtype: str
   name  math
0   Ali  78.0
1  Sara  92.0
2  Omar  65.0


> You may see `students.name` (dot style). Avoid it: it breaks for names with spaces or names like `count`.

### 3.2 `loc` — select by label (name)

In [16]:
print(students.loc[0])                           # row with index label 0
print(students.loc[0:2, ["name", "math"]])       # rows 0 to 2 - INCLUDING 2
print(students.loc[:, "name":"age"])             # all rows, columns from name to age

student_id            101
name                  Ali
gender                  M
province            Kabul
age                    20
study_hours             5
math                 78.0
science                74
english                80
enroll_date    2023-01-15
passed                yes
Name: 0, dtype: object
   name  math
0   Ali  78.0
1  Sara  92.0
2  Omar  65.0
      name gender  province  age
0      Ali      M     Kabul   20
1     Sara      F     Herat   22
2     Omar      M    kabul    19
3    Zahra      F     Balkh   21
4    Ahmad      M     Kabul   23
5   Fatima      F     HERAT   20
6    Hamid      M     Balkh   99
7   Maryam      F       NaN   22
8    Yusuf      M     Kabul   21
9    Laila      F     Herat   19
10    Omar      M    kabul    19
11   Nadia      F  Kandahar   24


### 3.3 `iloc` — select by position number

In [17]:
print(students.iloc[0])              # first row
print(students.iloc[0:2, 0:3])       # rows 0,1 and columns 0,1,2 - end NOT included
print(students.iloc[-1])             # last row

student_id            101
name                  Ali
gender                  M
province            Kabul
age                    20
study_hours             5
math                 78.0
science                74
english                80
enroll_date    2023-01-15
passed                yes
Name: 0, dtype: object
   student_id  name gender
0         101   Ali      M
1         102  Sara      F
student_id            112
name                Nadia
gender                  F
province         Kandahar
age                    24
study_hours             5
math                 72.0
science                70
english                68
enroll_date    2023-04-01
passed                yes
Name: 11, dtype: object


`loc` and `iloc` look the same here because our index is `0, 1, 2...`. The difference shows when the index has names or dates:

In [18]:
print(daily.loc["2023-01-02"])       # by date label
print(daily.iloc[1])                 # by position - same row

A    1.523030
B   -0.234153
C   -0.234137
Name: 2023-01-02 00:00:00, dtype: float64
A    1.523030
B   -0.234153
C   -0.234137
Name: 2023-01-02 00:00:00, dtype: float64


### 3.4 One single value — `at` and `iat`

In [19]:
print(students.at[1, "name"])
print(students.iat[1, 1])

Sara
Sara


### 3.5 Filtering rows with conditions

A condition like `students["math"] > 80` gives a column of `True`/`False`. Putting it inside `[]` keeps the `True` rows.

To combine conditions use **`&` (and)**, **`|` (or)**, **`~` (not)** — and put **each condition in brackets**.

In [20]:
print(students["math"] > 80)                              # True / False for each row
print(students[students["math"] > 80][["name", "math"]])

both = students[(students["math"] > 60) & (students["gender"] == "F")]
print(both[["name", "gender", "math"]])

either = students[(students["age"] < 20) | (students["study_hours"] >= 8)]
print(either[["name", "age", "study_hours"]])

0     False
1      True
2     False
3     False
4     False
5      True
6     False
7      True
8     False
9     False
10    False
11    False
Name: math, dtype: bool
     name  math
1    Sara  92.0
5  Fatima  95.0
7  Maryam  88.0
      name gender  math
1     Sara      F  92.0
5   Fatima      F  95.0
7   Maryam      F  88.0
11   Nadia      F  72.0
      name  age  study_hours
1     Sara   22            8
2     Omar   19            3
5   Fatima   20            9
9    Laila   19            6
10    Omar   19            3


**Handy filter helpers:**

| Helper | Meaning |
|---|---|
| `isin([...])` | value is in a list |
| `between(a, b)` | value is from a to b (inclusive) |
| `isna()` / `notna()` | value is missing / not missing |

In [21]:
print(students[students["province"].isin(["Herat", "Balkh"])][["name", "province"]])
print(students[students["age"].between(20, 22)][["name", "age"]])
print(students[students["math"].isna()][["name", "math"]])

    name province
1   Sara    Herat
3  Zahra    Balkh
6  Hamid    Balkh
9  Laila    Herat
     name  age
0     Ali   20
1    Sara   22
3   Zahra   21
5  Fatima   20
7  Maryam   22
8   Yusuf   21
    name  math
3  Zahra   NaN
9  Laila   NaN


### 3.6 `query()` — filtering in plain words

Write the condition as text. Inside `query` you use `and` / `or`, and `@` to use a Python variable.

In [22]:
print(students.query("math > 80 and gender == 'F'")[["name", "math"]])

limit = 7
print(students.query("study_hours >= @limit")[["name", "study_hours"]])

     name  math
1    Sara  92.0
5  Fatima  95.0
7  Maryam  88.0
     name  study_hours
1    Sara            8
5  Fatima            9
7  Maryam            7


## 4. Adding, Renaming & Dropping 🟢

| Task | Code |
|---|---|
| new column | `df["new"] = ...` |
| change values where a condition is true | `df.loc[condition, "col"] = value` |
| if / else column | `np.where(condition, value_if_true, value_if_false)` |
| rename columns | `df.rename(columns={"old": "new"})` |
| remove columns / rows | `df.drop(columns=[...])` / `df.drop(index=[...])` |
| use a column as the index | `df.set_index("col")` |
| turn the index back into a column | `df.reset_index()` |

Most methods **return a new table** — assign the result to keep it: `df = df.drop(...)`.

In [23]:
s2 = students.copy()      # work on a copy so the original stays unchanged

s2["total"] = s2["math"] + s2["science"] + s2["english"]       # from other columns
s2["school"] = "AAII"                                           # same value for every row
s2["hard_worker"] = np.where(s2["study_hours"] >= 6, "yes", "no")
s2.loc[s2["science"] >= 85, "award"] = "science star"           # other rows get NaN

print(s2[["name", "total", "school", "hard_worker", "award"]])

      name  total school hard_worker         award
0      Ali  232.0   AAII          no           NaN
1     Sara  266.0   AAII         yes  science star
2     Omar  195.0   AAII          no           NaN
3    Zahra    NaN   AAII         yes           NaN
4    Ahmad  173.0   AAII          no           NaN
5   Fatima  274.0   AAII         yes  science star
6    Hamid  211.0   AAII          no           NaN
7   Maryam  255.0   AAII         yes  science star
8    Yusuf  150.0   AAII          no           NaN
9    Laila    NaN   AAII         yes           NaN
10    Omar  195.0   AAII          no           NaN
11   Nadia  210.0   AAII          no           NaN


In [24]:
renamed = s2.rename(columns={"math": "math_score", "science": "science_score"})
print(renamed.columns.tolist())

smaller = s2.drop(columns=["school", "award"])     # remove columns
fewer = s2.drop(index=[0, 1])                      # remove rows with index 0 and 1
print(smaller.shape, fewer.shape)

by_id = s2.set_index("student_id")                 # student_id becomes the row label
print(by_id.loc[105, ["name", "total"]])
print(by_id.reset_index().columns[:3].tolist())    # back to a normal column

['student_id', 'name', 'gender', 'province', 'age', 'study_hours', 'math_score', 'science_score', 'english', 'enroll_date', 'passed', 'total', 'school', 'hard_worker', 'award']
(12, 13) (10, 15)
name     Ahmad
total    173.0
Name: 105, dtype: object
['student_id', 'name', 'gender']


## 5. Sorting 🟢

| Code | Does |
|---|---|
| `sort_values("col")` | sort rows by a column (smallest first) |
| `sort_values("col", ascending=False)` | largest first |
| `sort_values(["a", "b"])` | sort by `a`, then by `b` for ties |
| `sort_index()` | sort by the row labels |
| `nlargest(n, "col")` / `nsmallest(n, "col")` | top / bottom *n* rows |
| `rank()` | position of each value (1 = smallest) |

In [25]:
print(students.sort_values("science", ascending=False)[["name", "science"]].head(3))
print(students.sort_values(["gender", "english"], ascending=[True, False])[["name", "gender", "english"]])
print(students.nlargest(3, "english")[["name", "english"]])

r = students[["name", "science"]].copy()
r["rank"] = r["science"].rank(ascending=False)     # 1 = best score
print(r.sort_values("rank").head(4))

     name  science
5  Fatima       91
1    Sara       89
7  Maryam       85
      name gender  english
3    Zahra      F       90
5   Fatima      F       88
1     Sara      F       85
7   Maryam      F       82
9    Laila      F       79
11   Nadia      F       68
0      Ali      M       80
6    Hamid      M       75
2     Omar      M       70
10    Omar      M       70
4    Ahmad      M       60
8    Yusuf      M       55
     name  english
3   Zahra       90
5  Fatima       88
1    Sara       85
     name  science  rank
5  Fatima       91   1.0
1    Sara       89   2.0
7  Maryam       85   3.0
3   Zahra       80   4.0


## 6. Missing Data 🟢

**Missing data** = a value that was never recorded. Pandas shows it as `NaN` (numbers), `None` (text) or `NaT` (dates).

**Why it matters for AI:** most models **crash** or give wrong results with missing values. You must **find** them, then **remove** or **fill** them.

### 6.1 Finding missing values

In [26]:
print(students.isna().sum())                        # how many missing per column
print((students.isna().mean() * 100).round(1))      # % missing per column

student_id     0
name           0
gender         0
province       1
age            0
study_hours    0
math           2
science        0
english        0
enroll_date    0
passed         0
dtype: int64
student_id      0.0
name            0.0
gender          0.0
province        8.3
age             0.0
study_hours     0.0
math           16.7
science         0.0
english         0.0
enroll_date     0.0
passed          0.0
dtype: float64


### 6.2 Removing missing values — `dropna()`

| Code | Removes |
|---|---|
| `dropna()` | rows with **any** missing value |
| `dropna(subset=["math"])` | rows where **math** is missing |
| `dropna(how="all")` | rows where **everything** is missing |
| `dropna(axis=1)` | **columns** with missing values |

Only drop when few rows are affected — otherwise you throw away useful data.

In [27]:
print(students.dropna().shape)
print(students.dropna(subset=["math"]).shape)
print(students.dropna(axis=1).columns.tolist())

(9, 11)
(10, 11)
['student_id', 'name', 'gender', 'age', 'study_hours', 'science', 'english', 'enroll_date', 'passed']


### 6.3 Filling missing values — `fillna()`

| Strategy | Code | Good for |
|---|---|---|
| a fixed value | `fillna(0)` / `fillna("Unknown")` | counts, text |
| mean | `fillna(s.mean())` | numbers without extreme values |
| median | `fillna(s.median())` | numbers **with** extreme values (safer) |
| mode (most common) | `fillna(s.mode()[0])` | text categories |
| previous value | `ffill()` | time series |
| next value | `bfill()` | time series |
| in-between estimate | `interpolate()` | smooth sequences |

In [28]:
f = students.copy()
f["math"] = f["math"].fillna(f["math"].median())
f["province"] = f["province"].fillna("Unknown")
print(f[["name", "province", "math"]])
print("missing left:", f.isna().sum().sum())

      name  province  math
0      Ali     Kabul  78.0
1     Sara     Herat  92.0
2     Omar    kabul   65.0
3    Zahra     Balkh  71.0
4    Ahmad     Kabul  55.0
5   Fatima     HERAT  95.0
6    Hamid     Balkh  70.0
7   Maryam   Unknown  88.0
8    Yusuf     Kabul  45.0
9    Laila     Herat  71.0
10    Omar    kabul   65.0
11   Nadia  Kandahar  72.0
missing left: 0


In [29]:
# Different rule for each column in one go
f2 = students.fillna({"math": students["math"].mean(), "province": "Unknown"})
print(f2.isna().sum().sum())

0


In [30]:
readings = pd.Series([10, np.nan, np.nan, 40, np.nan, 60])
print(pd.DataFrame({
    "original": readings,
    "ffill": readings.ffill(),
    "bfill": readings.bfill(),
    "interpolate": readings.interpolate(),
}))

   original  ffill  bfill  interpolate
0      10.0   10.0   10.0         10.0
1       NaN   10.0   40.0         20.0
2       NaN   10.0   40.0         30.0
3      40.0   40.0   40.0         40.0
4       NaN   40.0   60.0         50.0
5      60.0   60.0   60.0         60.0


> **Common mistake:** `df["math"].fillna(0, inplace=True)` does **not** change `df` in new Pandas versions. Always write it as `df["math"] = df["math"].fillna(0)`.

## 7. Duplicates 🟢

**Duplicates** = the same record appearing more than once (often from joining files or typing data twice). In ML they can make results look better than they are.

| Code | Does |
|---|---|
| `duplicated()` | True for each repeated row |
| `drop_duplicates()` | removes repeated rows |
| `subset=["col"]` | only check certain columns (e.g. an ID) |
| `keep="first"` / `"last"` / `False` | which copy to keep (`False` removes all copies) |

In [31]:
print("duplicate rows:", students.duplicated().sum())
print(students[students.duplicated(keep=False)][["student_id", "name"]])   # show all copies

no_dups = students.drop_duplicates(subset=["student_id"])
print(students.shape, "->", no_dups.shape)

duplicate rows: 1
    student_id  name
2          103  Omar
10         103  Omar
(12, 11) -> (11, 11)


## 8. Basic Statistics 🟢
- We use statistical methods in pandas to understand data, find patterns, and make better decisions.

| Method | Meaning |
|---|---|
| `count()` | number of non-missing values |
| `sum()`, `mean()`, `median()` | total, average, middle value |
| `min()`, `max()` | smallest, largest |
| `std()` | how spread out the values are |
| `mode()` | most common value |
| `quantile(0.9)` | value below which 90% of data falls |
| `idxmax()` / `idxmin()` | **row label** of the max / min |

Missing values are **skipped automatically**.

In [32]:
print(students["math"].mean())
print(students["math"].median())
print(students["math"].max(), "scored by row", students["math"].idxmax())
print(students[["math", "science", "english"]].agg(["mean", "median", "std", "min", "max"]).round(1))

72.5
71.0
95.0 scored by row 5
        math  science  english
mean    72.5     71.7     75.2
median  71.0     72.0     77.0
std     16.1     13.2     10.9
min     45.0     50.0     55.0
max     95.0     91.0     90.0


### 8.1 `axis` — down or across?

| `axis=0` (default) | `axis=1` |
|---|---|
| calculates **down** each column | calculates **across** each row |
| average of each subject | each student's average |

- `axis=0` → rows are removed/combined
- `axis=1` → columns are removed/combined
```python
arr = [[1, 2, 3],
       [4, 5, 6]]
       
np.sum(arr, axis=0)  # Sum rows together → one result per column
# 1+4, 2+5, 3+6 = [5, 7, 9]

np.sum(arr, axis=1)  # Sum columns together → one result per row
# 1+2+3, 4+5+6 = [6, 15]
```

In [33]:
subjects = students[["math", "science", "english"]]
print(subjects.mean())                   # one number per column
print(subjects.mean(axis=1).round(1))    # one number per student

math       72.500000
science    71.666667
english    75.166667
dtype: float64
0     77.3
1     88.7
2     65.0
3     85.0
4     57.7
5     91.3
6     70.3
7     85.0
8     50.0
9     78.0
10    65.0
11    70.0
dtype: float64


### 8.2 Proportions, running totals and changes

In [34]:
print(students["passed"].value_counts(normalize=True))     # share of each value (0 to 1)

sales = pd.Series([100, 120, 90, 150])
print(pd.DataFrame({"sales": sales, "running_total": sales.cumsum(),
                    "change": sales.diff(), "pct_change": sales.pct_change().round(2)}))

passed
yes    0.833333
no     0.166667
Name: proportion, dtype: float64
   sales  running_total  change  pct_change
0    100            100     NaN         NaN
1    120            220    20.0        0.20
2     90            310   -30.0       -0.25
3    150            460    60.0        0.67


### 8.3 Correlation

**Correlation** = how two number columns are related.
- +1 → Both move together. If one increases, the other increases.
- 0 → No linear relationship between them.
- −1 → They move in opposite directions. If one increases, the other decreases.
  
**Example:**
- Study hours ↑, exam marks ↑ → positive correlation.
- Price ↑, demand ↓ → negative correlation (in many situations).
- Shoe size and exam marks → likely little or no correlation.

In [35]:
print(students[["study_hours", "math", "science", "english"]].corr().round(2))

             study_hours  math  science  english
study_hours         1.00  0.98     0.99     0.89
math                0.98  1.00     0.98     0.97
science             0.99  0.98     1.00     0.91
english             0.89  0.97     0.91     1.00


## ✅ Part 1 Recap — cleaning our dataset 🟢

Let's use what we learned to clean the student table. We'll use `students_clean` in Part 2.

In [37]:
students_clean = students.drop_duplicates(subset=["student_id"]).copy()          # 1. remove duplicate

students_clean["province"] = students_clean["province"].replace(                  # 2. fix messy text
    {" kabul ": "Kabul", "HERAT": "Herat"})
students_clean["province"] = students_clean["province"].fillna("Unknown")          # 3. fill missing text

students_clean.loc[students_clean["age"] > 60, "age"] = students_clean["age"].median()   # 4. fix impossible age
students_clean["math"] = students_clean["math"].fillna(students_clean["math"].median())  # 5. fill missing math

students_clean = students_clean.reset_index(drop=True)                             # 6. tidy index
print("missing:", students_clean.isna().sum().sum(), "| duplicates:", students_clean.duplicated().sum())
students_clean

missing: 0 | duplicates: 0


,student_id,name,gender,province,age,study_hours,math,science,english,enroll_date,passed
0,101,Ali,M,Kabul,20,5,78.0,74,80,2023-01-15,yes
1,102,Sara,F,Herat,22,8,92.0,89,85,2023-02-03,yes
2,103,Omar,M,Kabul,19,3,65.0,60,70,2023-01-20,yes
3,104,Zahra,F,Balkh,21,6,72.0,80,90,2023-03-11,yes
4,105,Ahmad,M,Kabul,23,2,55.0,58,60,2023-02-27,no
5,106,Fatima,F,Herat,20,9,95.0,91,88,2023-01-05,yes
6,107,Hamid,M,Balkh,21,4,70.0,66,75,2023-04-18,yes
7,108,Maryam,F,Unknown,22,7,88.0,85,82,2023-03-02,yes
8,109,Yusuf,M,Kabul,21,1,45.0,50,55,2023-05-09,no
9,110,Laila,F,Herat,19,6,72.0,77,79,2023-02-14,yes


---
# 🟡 PART 2 — INTERMEDIATE

## 9. Data Types & Converting Them 🟡

Every column has a **dtype** (data type). A wrong type causes bugs: numbers stored as text can't be averaged, dates stored as text can't be compared.

| dtype | Holds | Example |
|---|---|---|
| `int64` | whole numbers | age |
| `float64` | decimals (and any number column with `NaN`) | score |
| `bool` | True / False | is_active |
| `object` / `str` | text (`object` in Pandas 2, `str` in Pandas 3) | name |
| `category` | text with few repeating values — saves memory | province |
| `datetime64` | dates and times | enroll_date |
| `Int64` (capital I) | whole numbers that **can** have missing values | count with gaps |

### 9.1 `astype()` — change a type

In [38]:
t = pd.DataFrame({"a": ["1", "2", "3"], "b": [1.7, 2.2, 3.9]})
print(t.dtypes)

t["a"] = t["a"].astype(int)      # text -> number
t["b"] = t["b"].astype(int)      # careful: 1.7 becomes 1 (cut off, NOT rounded)
print(t)
print(t.dtypes)

a        str
b    float64
dtype: object
   a  b
0  1  1
1  2  2
2  3  3
a    int64
b    int64
dtype: object


### 9.2 Converting messy numbers and dates

- `pd.to_numeric(..., errors="coerce")` — turns what it can into numbers; bad values like `"n/a"` become `NaN` instead of crashing.
- `pd.to_datetime(...)` — turns text into real dates.

In [39]:
dirty = pd.Series(["12", "7.5", "n/a", "?", "30"])
print(pd.to_numeric(dirty, errors="coerce"))

students_clean["enroll_date"] = pd.to_datetime(students_clean["enroll_date"])
print(students_clean["enroll_date"].dtype)

0    12.0
1     7.5
2     NaN
3     NaN
4    30.0
dtype: float64
datetime64[us]


### 9.3 Category and nullable integers

In [40]:
students_clean["province"] = students_clean["province"].astype("category")
print(students_clean["province"].dtype, students_clean["province"].cat.categories.tolist())

print(pd.Series([1, 2, None]).dtype)                  # float64 - integers became decimals
print(pd.Series([1, 2, None], dtype="Int64"))          # stays whole numbers, missing shown as <NA>

category ['Balkh', 'Herat', 'Kabul', 'Kandahar', 'Unknown']
float64
0       1
1       2
2    <NA>
dtype: Int64


### 9.4 `select_dtypes()` — pick columns by type

Useful in ML: number columns get scaled, text columns get encoded.

In [41]:
print(students_clean.select_dtypes(include="number").columns.tolist())
print(students_clean.select_dtypes(exclude="number").columns.tolist())

['student_id', 'age', 'study_hours', 'math', 'science', 'english']
['name', 'gender', 'province', 'enroll_date', 'passed']


## 10. Text (String) Operations 🟡

Add `.str` after a text column to use string methods on **every row at once**. Missing values stay missing instead of crashing.

| Method | Does |
|---|---|
| `str.strip()` | remove spaces at both ends |
| `str.lower()` / `upper()` / `title()` | change letter case |
| `str.len()` | length |
| `str.contains("x")` | True if text contains "x" |
| `str.startswith()` / `endswith()` | starts / ends with |
| `str.replace(a, b)` | replace text |
| `str.split(sep)` | split into parts |
| `str.extract(pattern)` | pull out a part using a regular expression |

In [42]:
messy = pd.Series([" Kabul ", "HERAT", "mazar-e-sharif", None])
print(messy.str.strip())
print(messy.str.strip().str.title())
print(messy.str.replace("-", " "))
print(messy.str.len())
print(messy.str.contains("a", case=False, na=False))    # na=False: missing counts as False

0             Kabul
1             HERAT
2    mazar-e-sharif
3               NaN
dtype: str
0             Kabul
1             Herat
2    Mazar-E-Sharif
3               NaN
dtype: str
0            Kabul 
1             HERAT
2    mazar e sharif
3               NaN
dtype: str
0     7.0
1     5.0
2    14.0
3     NaN
dtype: float64
0     True
1     True
2     True
3    False
dtype: bool


**The standard fix for messy text:** `str.strip()` then `str.title()`. This fixes our raw data in one line (compare with the `replace` dictionary we used in the recap):

In [43]:
print(students["province"].str.strip().str.title().value_counts())

province
Kabul       5
Herat       3
Balkh       2
Kandahar    1
Name: count, dtype: int64


**Splitting into columns and extracting parts:**

In [44]:
people = pd.DataFrame({
    "full_name": ["Ali Rahimi", "Sara Ahmadi", "Omar Karimi"],
    "email": ["ali@aaii.org", "sara@gmail.com", "omar@aaii.org"],
    "course": ["AI-101 (2024)", "DS-202 (2025)", "ML-303 (2024)"],
})
people[["first", "last"]] = people["full_name"].str.split(" ", expand=True)   # expand=True -> columns
people["domain"] = people["email"].str.split("@").str[1]                      # part after @
people["year"] = people["course"].str.extract(r"\((\d{4})\)")                 # 4 digits inside ( )
print(people)

     full_name           email         course first    last     domain  year
0   Ali Rahimi    ali@aaii.org  AI-101 (2024)   Ali  Rahimi   aaii.org  2024
1  Sara Ahmadi  sara@gmail.com  DS-202 (2025)  Sara  Ahmadi  gmail.com  2025
2  Omar Karimi   omar@aaii.org  ML-303 (2024)  Omar  Karimi   aaii.org  2024


**Cleaning text for NLP (language AI):** lower case → remove symbols → remove extra spaces. Word count is a simple useful feature.

In [45]:
reviews = pd.Series(["Great course!!  Loved it.", "too FAST, hard to follow...", "Clear & useful :)"])
clean = (reviews.str.lower()
                .str.replace(r"[^a-z\s]", "", regex=True)    # keep only letters and spaces
                .str.replace(r"\s+", " ", regex=True)        # many spaces -> one space
                .str.strip())
print(pd.DataFrame({"raw": reviews, "clean": clean, "words": clean.str.split().str.len()}))

                           raw                    clean  words
0    Great course!!  Loved it.    great course loved it      4
1  too FAST, hard to follow...  too fast hard to follow      5
2            Clear & useful :)             clear useful      2


## 11. Applying Functions 🟡

| Method | Use it to |
|---|---|
| `Series.map(dict)` | swap values using a lookup table |
| `Series.apply(func)` | run your own function on each value |
| `DataFrame.apply(func, axis=1)` | run a function on each **row** (can use several columns) |
| `DataFrame.map(func)` | run a function on **every cell** (older Pandas: `applymap`) |
| `np.where(cond, a, b)` | fast if / else |
| `np.select([conds], [values], default)` | fast if / elif / else |

In [46]:
sc = students_clean[["name", "gender", "math", "science"]].copy()

sc["gender_full"] = sc["gender"].map({"M": "Male", "F": "Female"})
sc["math_curved"] = sc["math"].apply(lambda x: min(x + 5, 100))
sc["best"] = sc.apply(lambda row: "math" if row["math"] >= row["science"] else "science", axis=1)
print(sc.head())

print(sc[["math", "science"]].head(3).map(lambda v: f"{v:.0f}%"))

    name gender  math  science gender_full  math_curved     best
0    Ali      M  78.0       74        Male         83.0     math
1   Sara      F  92.0       89      Female         97.0     math
2   Omar      M  65.0       60        Male         70.0     math
3  Zahra      F  72.0       80      Female         77.0  science
4  Ahmad      M  55.0       58        Male         60.0  science
  math science
0  78%     74%
1  92%     89%
2  65%     60%


In [47]:
avg = students_clean[["math", "science", "english"]].mean(axis=1)
conditions = [avg >= 85, avg >= 70, avg >= 55]
grades = ["A", "B", "C"]
students_clean["grade"] = np.select(conditions, grades, default="F")
print(students_clean[["name", "grade"]].assign(avg=avg.round(1)))

      name grade   avg
0      Ali     B  77.3
1     Sara     A  88.7
2     Omar     C  65.0
3    Zahra     B  80.7
4    Ahmad     C  57.7
5   Fatima     A  91.3
6    Hamid     B  70.3
7   Maryam     A  85.0
8    Yusuf     F  50.0
9    Laila     B  76.0
10   Nadia     B  70.0


### 11.1 Vectorization — why built-in operations are faster

**Vectorization** = working on a whole column at once with fast built-in code. `apply` runs a slow Python loop underneath.

**Rule:** use `+ - * /`, `np.where`, `.str`, `.dt` when you can. Use `apply` only when there's no built-in way.

In [48]:
import time

big = pd.DataFrame({"x": np.random.randn(500_000)})

start = time.time()
big["y_apply"] = big["x"].apply(lambda v: v * 2 + 1)
t_apply = time.time() - start

start = time.time()
big["y_vector"] = big["x"] * 2 + 1
t_vec = time.time() - start

print(f"apply      : {t_apply:.4f} s")
print(f"vectorized : {t_vec:.4f} s   (about {t_apply / t_vec:.0f}x faster)")

apply      : 0.1969 s
vectorized : 0.0055 s   (about 36x faster)


## 12. Outliers & Binning 🟡

### 12.1 Outliers

An **outlier** is a value far away from the others — a real extreme or a mistake (like age 99). Outliers pull the mean up/down and confuse many models.

**IQR rule** (most common):
- Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 − Q1
- anything below **Q1 − 1.5×IQR** or above **Q3 + 1.5×IQR** is an outlier

**Z-score rule:** z = (value − mean) / std. |z| > 3 is an outlier (needs lots of roughly bell-shaped data).

In [49]:
age = students["age"]                          # raw data, still has 99
q1, q3 = age.quantile(0.25), age.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"normal range: {low} to {high}")
print(students[(age < low) | (age > high)][["name", "age"]])

normal range: 16.0 to 26.0
    name  age
6  Hamid   99


**What to do with outliers:** remove the row, replace it (e.g. with the median), or **cap** it with `clip()`.

In [50]:
print(age.clip(lower=low, upper=high).tolist())      # 99 is capped to the upper limit

[20, 22, 19, 21, 23, 20, 26, 22, 21, 19, 19, 24]


### 12.2 Binning — numbers into groups

| Function | Groups by |
|---|---|
| `pd.cut()` | ranges **you choose** (e.g. 0–50 = fail) |
| `pd.qcut()` | equal-sized groups (each has about the same number of rows) |

In [51]:
students_clean["math_band"] = pd.cut(students_clean["math"], bins=[0, 50, 70, 85, 100],
                                     labels=["fail", "pass", "merit", "distinction"])
students_clean["effort"] = pd.qcut(students_clean["study_hours"], q=3, labels=["low", "medium", "high"])
print(students_clean[["name", "math", "math_band", "study_hours", "effort"]])

      name  math    math_band  study_hours  effort
0      Ali  78.0        merit            5  medium
1     Sara  92.0  distinction            8    high
2     Omar  65.0         pass            3     low
3    Zahra  72.0        merit            6  medium
4    Ahmad  55.0         pass            2     low
5   Fatima  95.0  distinction            9    high
6    Hamid  70.0         pass            4     low
7   Maryam  88.0  distinction            7    high
8    Yusuf  45.0         fail            1     low
9    Laila  72.0        merit            6  medium
10   Nadia  72.0        merit            5  medium


## 13. Encoding Categories 🟡

Models need **numbers**, not text like `"Kabul"`. **Encoding** turns categories into numbers.

| Method | How | Use when |
|---|---|---|
| **Mapping / ordinal** | each category → an integer | categories have an **order** (small < medium < large) |
| **One-hot** | one 0/1 column per category | **no order** (provinces, colors) |
| **Binary** | two values → 0 / 1 | yes/no, M/F |

**Why not just number provinces 0, 1, 2?** The model would think Balkh (2) is "twice" Herat (1) — which means nothing. One-hot avoids that.

In [52]:
sizes = pd.Series(["small", "medium", "large", "small"])
print(sizes.map({"small": 0, "medium": 1, "large": 2}).tolist())      # ordinal

students_clean["passed_num"] = students_clean["passed"].map({"no": 0, "yes": 1})   # binary
print(students_clean[["passed", "passed_num"]].head(4))

[0, 1, 2, 0]
  passed  passed_num
0    yes           1
1    yes           1
2    yes           1
3    yes           1


In [53]:
grades_df = pd.DataFrame({"grade": ["A", "B", "A", "C"]})
print(pd.get_dummies(grades_df, columns=["grade"], dtype=int))
print(pd.get_dummies(grades_df, columns=["grade"], drop_first=True, dtype=int))

   grade_A  grade_B  grade_C
0        1        0        0
1        0        1        0
2        1        0        0
3        0        0        1
   grade_B  grade_C
0        0        0
1        1        0
2        0        0
3        0        1


**`drop_first=True`** removes one column: with 3 categories, 2 columns are enough (if both are 0, it must be the third). This avoids the **dummy variable trap**, which confuses linear models.

`replace()` swaps values anywhere in a table — useful for text fixes:

In [54]:
print(grades_df.replace({"A": "excellent", "B": "good", "C": "average"}))

       grade
0  excellent
1       good
2  excellent
3    average


## 14. Combining Tables 🟡

| Tool | Does | Think of it as |
|---|---|---|
| `pd.concat` | stacks tables on top of each other (or side by side) | gluing |
| `pd.merge` | joins rows that share a **key** value | SQL JOIN |
| `df.join` | merges using the **index** | shortcut for merge |

### 14.1 `concat`

In [55]:
jan = pd.DataFrame({"student": ["Ali", "Sara"], "score": [80, 90]})
feb = pd.DataFrame({"student": ["Omar", "Zahra"], "score": [70, 88]})

print(pd.concat([jan, feb], ignore_index=True))       # stacked, fresh index 0..3
print(pd.concat([jan, feb], axis=1))                  # side by side

  student  score
0     Ali     80
1    Sara     90
2    Omar     70
3   Zahra     88
  student  score student  score
0     Ali     80    Omar     70
1    Sara     90   Zahra     88


### 14.2 `merge` — the four join types

| `how=` | Keeps |
|---|---|
| `"inner"` (default) | only keys found in **both** tables |
| `"left"` | **all** rows from the left table |
| `"right"` | **all** rows from the right table |
| `"outer"` | **all** rows from **both** tables |

Missing matches are filled with `NaN`.

In [56]:
info = pd.DataFrame({"id": [1, 2, 3, 4], "name": ["Ali", "Sara", "Omar", "Zahra"]})
exam = pd.DataFrame({"id": [2, 3, 4, 5], "score": [88, 72, 95, 60]})

for how in ["inner", "left", "right", "outer"]:
    print(f"--- how='{how}' ---")
    print(pd.merge(info, exam, on="id", how=how))

--- how='inner' ---
   id   name  score
0   2   Sara     88
1   3   Omar     72
2   4  Zahra     95
--- how='left' ---
   id   name  score
0   1    Ali    NaN
1   2   Sara   88.0
2   3   Omar   72.0
3   4  Zahra   95.0
--- how='right' ---
   id   name  score
0   2   Sara     88
1   3   Omar     72
2   4  Zahra     95
3   5    NaN     60
--- how='outer' ---
   id   name  score
0   1    Ali    NaN
1   2   Sara   88.0
2   3   Omar   72.0
3   4  Zahra   95.0
4   5    NaN   60.0


**Useful merge options:**

| Option | Use |
|---|---|
| `left_on=` / `right_on=` | key columns have different names |
| `suffixes=` | rename columns that exist in both tables |
| `indicator=True` | adds `_merge` column showing where each row came from — great for checking |
| `validate="one_to_one"` | error if keys aren't unique as expected (stops accidental duplicates) |

In [57]:
courses = pd.DataFrame({"student_id": [1, 1, 2, 5], "course": ["AI", "Python", "AI", "Stats"]})
print(pd.merge(info, courses, left_on="id", right_on="student_id", how="outer",
               indicator=True, validate="one_to_many"))

mid = pd.DataFrame({"id": [1, 2], "score": [80, 90]})
final = pd.DataFrame({"id": [1, 2], "score": [85, 70]})
print(pd.merge(mid, final, on="id", suffixes=("_mid", "_final")))

    id   name  student_id  course      _merge
0  1.0    Ali         1.0      AI        both
1  1.0    Ali         1.0  Python        both
2  2.0   Sara         2.0      AI        both
3  3.0   Omar         NaN     NaN   left_only
4  4.0  Zahra         NaN     NaN   left_only
5  NaN    NaN         5.0   Stats  right_only
   id  score_mid  score_final
0   1         80           85
1   2         90           70


### 14.3 `join` — merge on the index

In [58]:
left = pd.DataFrame({"lval": [1, 2]}, index=["foo", "bar"])
right = pd.DataFrame({"rval": [4, 5]}, index=["foo", "bar"])
print(left.join(right))

     lval  rval
foo     1     4
bar     2     5


## 15. GroupBy 🟡

**GroupBy = Split → Apply → Combine.** It answers questions like *"what is the average score **per province**?"*

1. **Split** the rows into groups (one group per province)
2. **Apply** a calculation to each group (mean)
3. **Combine** the results into a small table

In [59]:
sc = students_clean
print(sc.groupby("province", observed=True)["math"].mean().round(1))
print(sc.groupby("gender")[["math", "science"]].mean().round(1))
print(sc.groupby(["province", "gender"], observed=True).size())       # rows per group

province
Balkh       71.0
Herat       86.3
Kabul       60.8
Kandahar    72.0
Unknown     88.0
Name: math, dtype: float64
        math  science
gender               
F       81.8     82.0
M       62.6     61.6
province  gender
Balkh     F         1
          M         1
Herat     F         3
Kabul     M         4
Kandahar  F         1
Unknown   F         1
dtype: int64


> `observed=True` is used because `province` is a *category* column — it tells Pandas to show only groups that actually exist.

### 15.1 Several calculations at once — `agg`

**Named aggregation** gives clean column names: `new_name=("column", "calculation")`.

In [60]:
report = sc.groupby("gender").agg(
    students=("student_id", "count"),
    avg_math=("math", "mean"),
    best_english=("english", "max"),
    pass_rate=("passed_num", "mean"),
).round(2)
print(report)

        students  avg_math  best_english  pass_rate
gender                                             
F              6     81.83            90        1.0
M              5     62.60            80        0.6


### 15.2 `transform` and `filter`

| Method | Result |
|---|---|
| `agg` | **one row per group** |
| `transform` | **same number of rows as the original** — each row gets its group's value |
| `filter` | keeps only the **groups** that pass a test |

`transform` is great for ML features like *"how does this student compare with their gender's average?"*

In [61]:
t = sc[["name", "gender", "math"]].copy()
t["gender_avg_math"] = t.groupby("gender")["math"].transform("mean").round(1)
t["vs_avg"] = (t["math"] - t["gender_avg_math"]).round(1)
print(t)

print(sc.groupby("province", observed=True).filter(lambda g: len(g) >= 3)[["name", "province"]])

      name gender  math  gender_avg_math  vs_avg
0      Ali      M  78.0             62.6    15.4
1     Sara      F  92.0             81.8    10.2
2     Omar      M  65.0             62.6     2.4
3    Zahra      F  72.0             81.8    -9.8
4    Ahmad      M  55.0             62.6    -7.6
5   Fatima      F  95.0             81.8    13.2
6    Hamid      M  70.0             62.6     7.4
7   Maryam      F  88.0             81.8     6.2
8    Yusuf      M  45.0             62.6   -17.6
9    Laila      F  72.0             81.8    -9.8
10   Nadia      F  72.0             81.8    -9.8
     name province
0     Ali    Kabul
1    Sara    Herat
2    Omar    Kabul
4   Ahmad    Kabul
5  Fatima    Herat
8   Yusuf    Kabul
9   Laila    Herat


**Group-wise filling of missing values** (more accurate than one overall median):

In [62]:
raw_math = students.drop_duplicates(subset="student_id")[["name", "gender", "math"]].copy()
raw_math["math_filled"] = raw_math["math"].fillna(raw_math.groupby("gender")["math"].transform("median"))
print(raw_math[raw_math["math"].isna()])

    name gender  math  math_filled
3  Zahra      F   NaN         90.0
9  Laila      F   NaN         90.0


### 15.3 `crosstab` — count two categories against each other

In [63]:
print(pd.crosstab(sc["gender"], sc["passed"]))
print(pd.crosstab(sc["gender"], sc["passed"], normalize="index").round(2))     # row percentages

passed  no  yes
gender         
F        0    6
M        2    3
passed   no  yes
gender          
F       0.0  1.0
M       0.4  0.6


## 16. Reshaping 🟡

**Wide** vs **long** — the same data, two layouts:

| Wide | Long |
|---|---|
| one row per student, one column per subject | one row per student **per subject** |
| `student, math, science` | `student, subject, score` |
| good for ML | good for plotting and groupby |

| Tool | Converts |
|---|---|
| `melt` | wide → long |
| `pivot` | long → wide |
| `pivot_table` | long → wide **and** calculates (e.g. mean) when there are repeats |
| `stack` / `unstack` | columns ↔ inner row labels |
| `explode` | a list in a cell → one row per item |

In [64]:
wide = pd.DataFrame({"student": ["Ali", "Sara"], "math": [80, 90], "science": [70, 85]})
long = pd.melt(wide, id_vars="student", var_name="subject", value_name="score")
print(long)
print(long.pivot(index="student", columns="subject", values="score"))

  student  subject  score
0     Ali     math     80
1    Sara     math     90
2     Ali  science     70
3    Sara  science     85
subject  math  science
student               
Ali        80       70
Sara       90       85


In [65]:
print(pd.pivot_table(sc, values="math", index="province", columns="gender",
                     aggfunc="mean", observed=True).round(1))

gender       F     M
province            
Balkh     72.0  70.0
Herat     86.3   NaN
Kabul      NaN  60.8
Kandahar  72.0   NaN
Unknown   88.0   NaN


### 16.1 MultiIndex — rows with two levels of labels

Grouping by two columns gives a **MultiIndex**. Select with `.loc`, or flatten it with `reset_index()`.

In [66]:
multi = sc.groupby(["gender", "grade"])["math"].mean()
print(multi)
print(multi.loc["F"])              # the outer level
print(multi.unstack())             # inner level -> columns (wide)
print(multi.reset_index())         # back to normal columns

gender  grade
F       A        91.666667
        B        72.000000
M       B        74.000000
        C        60.000000
        F        45.000000
Name: math, dtype: float64
grade
A    91.666667
B    72.000000
Name: math, dtype: float64
grade           A     B     C     F
gender                             
F       91.666667  72.0   NaN   NaN
M             NaN  74.0  60.0  45.0
  gender grade       math
0      F     A  91.666667
1      F     B  72.000000
2      M     B  74.000000
3      M     C  60.000000
4      M     F  45.000000


In [67]:
tags = pd.DataFrame({"course": ["AI-101", "DS-202"], "tags": [["python", "ml"], ["stats", "pandas", "viz"]]})
print(tags.explode("tags", ignore_index=True))

   course    tags
0  AI-101  python
1  AI-101      ml
2  DS-202   stats
3  DS-202  pandas
4  DS-202     viz


## 17. Dates & Time Series 🟡

**Time series** = values recorded over time (daily sales, temperatures).

| Task | Tool |
|---|---|
| make a list of dates | `pd.date_range(start, periods, freq)` |
| text → date | `pd.to_datetime()` |
| parts of a date | `.dt.year`, `.dt.month`, `.dt.day_name()`, `.dt.dayofweek` |
| change the time step (daily → weekly) | `resample("W")` |
| previous value | `shift(1)` |
| moving average | `rolling(7).mean()` |
| running average from the start | `expanding().mean()` |
| average that favours recent values | `ewm(span=7).mean()` |

Frequency codes: `"D"` day, `"W"` week, `"MS"` month start, `"h"` hour.

### 17.1 Date parts

In [68]:
d = students_clean[["name", "enroll_date"]].copy()
d["year"] = d["enroll_date"].dt.year
d["month"] = d["enroll_date"].dt.month
d["weekday"] = d["enroll_date"].dt.day_name()
d["days_since"] = (pd.Timestamp("2024-01-01") - d["enroll_date"]).dt.days
d["deadline"] = d["enroll_date"] + pd.Timedelta(days=30)
print(d.head())

print(pd.to_datetime(pd.Series(["15/02/2023", "31/12/2023"]), format="%d/%m/%Y"))   # day/month format
print(pd.to_datetime(pd.Series(["2023-01-05", "not a date"]), errors="coerce"))      # bad -> NaT

    name enroll_date  year  month   weekday  days_since   deadline
0    Ali  2023-01-15  2023      1    Sunday         351 2023-02-14
1   Sara  2023-02-03  2023      2    Friday         332 2023-03-05
2   Omar  2023-01-20  2023      1    Friday         346 2023-02-19
3  Zahra  2023-03-11  2023      3  Saturday         296 2023-04-10
4  Ahmad  2023-02-27  2023      2    Monday         308 2023-03-29
0   2023-02-15
1   2023-12-31
dtype: datetime64[us]
0   2023-01-05
1          NaT
dtype: datetime64[us]


### 17.2 A daily sales series

In [69]:
days = pd.date_range("2023-01-01", periods=90, freq="D")
sales = pd.DataFrame({"sales": (100 + np.arange(90) * 0.8 + np.random.normal(0, 8, 90)).round(1)}, index=days)

print(sales.loc["2023-02"].head(3))              # all rows from February
print(sales.resample("W").sum().head(3))         # weekly totals
print(sales.resample("MS").mean().round(1))      # monthly averages

            sales
2023-02-01  127.6
2023-02-02  112.6
2023-02-03  129.1
            sales
2023-01-01   89.7
2023-01-08  740.0
2023-01-15  769.8
            sales
2023-01-01  111.1
2023-02-01  135.1
2023-03-01  158.2


### 17.3 Lags and moving windows

- `shift(1)` → **yesterday's** value on today's row. These "lag" columns let a model predict today from the past.
- `rolling(7)` → a 7-day **moving window**.
- The first rows get `NaN` because there is no earlier data yet.

In [70]:
fe = sales.copy()
fe["yesterday"] = fe["sales"].shift(1)
fe["change"] = fe["sales"].diff()
fe["avg_7d"] = fe["sales"].rolling(7).mean().round(1)
fe["avg_so_far"] = fe["sales"].expanding().mean().round(1)
fe["ewm_7"] = fe["sales"].ewm(span=7).mean().round(1)
print(fe.head(9))

            sales  yesterday  change  avg_7d  avg_so_far  ewm_7
2023-01-01   89.7        NaN     NaN     NaN        89.7   89.7
2023-01-02  115.7       89.7    26.0     NaN       102.7  104.6
2023-01-03  101.2      115.7   -14.5     NaN       102.2  103.1
2023-01-04   91.6      101.2    -9.6     NaN        99.6   98.9
2023-01-05  109.4       91.6    17.8     NaN       101.5  102.3
2023-01-06  102.2      109.4    -7.2     NaN       101.6  102.3
2023-01-07  118.0      102.2    15.8   104.0       104.0  106.8
2023-01-08  101.9      118.0   -16.1   105.7       103.7  105.5
2023-01-09   99.9      101.9    -2.0   103.5       103.3  104.0


## 18. Plotting 🟡

Pandas can draw charts directly: `df.plot(kind=...)` (it uses Matplotlib underneath — see the Matplotlib notebook for full control).

| kind | Shows |
|---|---|
| `line` | trend over time |
| `bar` / `barh` | compare categories |
| `hist` | distribution of one column |
| `box` | spread and outliers |
| `scatter` | relationship of two columns |

In [71]:
import matplotlib.pyplot as plt

sales["sales"].plot(title="Daily sales", figsize=(8, 3))
plt.show()

sc.groupby("gender")["math"].mean().plot(kind="bar", title="Average math by gender")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sc["math"].plot(kind="hist", bins=6, ax=axes[0], title="Math distribution")
students[["age"]].plot(kind="box", ax=axes[1], title="Age (raw) - spot the outlier")
sc.plot(kind="scatter", x="study_hours", y="math", ax=axes[2], title="Study hours vs math")
plt.tight_layout()
plt.show()

## 19. Reading & Saving Files 🟡

| Format | Read | Save | Notes |
|---|---|---|---|
| CSV | `pd.read_csv` | `to_csv` | most common, plain text |
| Excel | `pd.read_excel` | `to_excel` | needs `openpyxl` |
| JSON | `pd.read_json` | `to_json` | web / API data |
| Parquet | `pd.read_parquet` | `to_parquet` | fast & small, for big data; needs `pyarrow` |
| SQL | `pd.read_sql` | `to_sql` | databases |

> Use `index=False` when saving CSVs, or the index is saved as an extra column.

### 19.1 CSV

In [ ]:
loaded = pd.read_csv("students.csv")
print(loaded.shape)

loaded = pd.read_csv(
    "students.csv",
    usecols=["student_id", "name", "math", "enroll_date"],   # only these columns
    parse_dates=["enroll_date"],                             # convert to dates while loading
    na_values=["?", "n/a"],                                  # extra values that mean "missing"
    nrows=5,                                                 # only the first 5 rows
)
print(loaded)
print(loaded.dtypes)

**Very big files:** `chunksize` reads the file in pieces so it fits in memory.

In [ ]:
total = 0
for chunk in pd.read_csv("students.csv", chunksize=5):
    total += len(chunk)
print("rows read in chunks:", total)

### 19.2 JSON, Excel, Parquet

In [ ]:
report.to_json("report.json")
print(pd.read_json("report.json"))

nested = [{"id": 1, "user": {"name": "Ali", "city": "Kabul"}},
          {"id": 2, "user": {"name": "Sara", "city": "Herat"}}]
print(pd.json_normalize(nested))           # flattens nested JSON into columns

In [ ]:
try:
    sc.to_excel("students.xlsx", index=False)
    print(pd.read_excel("students.xlsx").head(3))
except ImportError:
    print("Install openpyxl: pip install openpyxl")

try:
    sc.to_parquet("students.parquet")
    print(pd.read_parquet("students.parquet").dtypes.head(4))     # keeps the dtypes!
except ImportError:
    print("Install pyarrow: pip install pyarrow")

### 19.3 SQL database

`sqlite3` comes with Python, so this runs anywhere.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")                 # a temporary database in memory
students[["student_id", "name", "gender", "math"]].to_sql("students", conn, index=False)

print(pd.read_sql("SELECT gender, AVG(math) AS avg_math FROM students GROUP BY gender", conn))
conn.close()

---
# 🔴 PART 3 — ADVANCED

## 20. Views, Copies & Copy-on-Write 🔴

When you take part of a table and change it, does the original change too? This used to cause the famous `SettingWithCopyWarning`.

**In Pandas 3 (Copy-on-Write):** a selected part always acts like a **separate copy**. Changing it never changes the original.

**Safe rules for every version:**
1. To change the original, use **one** `loc`: `df.loc[condition, "col"] = value`
2. To work on a separate part, add `.copy()`
3. Never "chain" like `df["col"][0] = 5` — in Pandas 3 it silently does nothing

In [ ]:
original = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

part = original[original["a"] > 1].copy()
part["b"] = 0
print(original)                              # unchanged

original.loc[original["a"] > 1, "b"] = 0     # the correct way to change the original
print(original)

## 21. Method Chaining & `pipe()` 🔴

**Method chaining** = writing several steps as one pipeline, instead of many temporary variables. Put it in brackets `( ... )` so each step can go on its own line.

- `assign(new_col=lambda d: ...)` adds columns inside a chain (`d` is the table at that step)
- `pipe(my_function)` puts **your own function** into the chain

In [ ]:
def fix_age(df, max_age=60):
    df = df.copy()
    df.loc[df["age"] > max_age, "age"] = df["age"].median()
    return df

cleaned = (
    pd.read_csv("students.csv", parse_dates=["enroll_date"])
      .drop_duplicates(subset="student_id")
      .assign(province=lambda d: d["province"].str.strip().str.title().fillna("Unknown"),
              math=lambda d: d["math"].fillna(d["math"].median()))
      .pipe(fix_age)
      .assign(average=lambda d: d[["math", "science", "english"]].mean(axis=1).round(1))
      .sort_values("average", ascending=False)
      .reset_index(drop=True)
)
cleaned.head()

## 22. Speed & Memory 🔴

| Tip | Why |
|---|---|
| Use vectorized operations, not loops or `apply` | 10–100× faster |
| If you must loop, use `itertuples()` not `iterrows()` | `iterrows` is very slow |
| Don't add rows to a DataFrame inside a loop | collect in a list, build the DataFrame once |
| Use `category` for repeating text | stores each label once |
| Use smaller number types (`float32`, `int8`) | half the memory or less |
| Load only needed columns (`usecols`) | less memory from the start |

In [ ]:
loop_df = pd.DataFrame({"a": np.arange(50_000), "b": np.arange(50_000)})

start = time.time(); total = 0
for _, row in loop_df.iterrows():
    total += row["a"] * row["b"]
t1 = time.time() - start

start = time.time(); total = 0
for row in loop_df.itertuples():
    total += row.a * row.b
t2 = time.time() - start

start = time.time()
total = (loop_df["a"] * loop_df["b"]).sum()
t3 = time.time() - start

print(f"iterrows   : {t1:.3f} s\nitertuples : {t2:.3f} s\nvectorized : {t3:.4f} s")

In [ ]:
# Right way to build a table in a loop (e.g. logging training results)
rows = []
for epoch in range(5):
    rows.append({"epoch": epoch, "loss": round(1 / (epoch + 1), 3)})
log = pd.DataFrame(rows)       # build once, at the end
print(log)

In [ ]:
large = pd.DataFrame({
    "category": np.random.choice(["A", "B", "C"], size=100_000),
    "value": np.random.randn(100_000),
    "count": np.random.randint(0, 100, size=100_000),
})
before = large.memory_usage(deep=True).sum() / 1024

large["category"] = large["category"].astype("category")
large["value"] = large["value"].astype("float32")
large["count"] = pd.to_numeric(large["count"], downcast="integer")    # becomes int8

after = large.memory_usage(deep=True).sum() / 1024
print(large.dtypes)
print(f"Before: {before:,.0f} KB  ->  After: {after:,.0f} KB  (saved {1 - after / before:.0%})")

## 23. Common Mistakes 🔴

| Mistake | What happens | Fix |
|---|---|---|
| `df[df.a > 1 and df.b < 5]` | error: "truth value is ambiguous" | use `&` `\|` `~` |
| `df[df.a > 1 & df.b < 5]` | wrong result | brackets: `(df.a > 1) & (df.b < 5)` |
| `df["c"][0] = 5` | does nothing in Pandas 3 | `df.loc[0, "c"] = 5` |
| `df["c"].fillna(0, inplace=True)` | does nothing in Pandas 3 | `df["c"] = df["c"].fillna(0)` |
| `df.dropna()` on its own line | `df` not changed | `df = df.dropna()` |
| thinking `loc[0:2]` gives 2 rows | gives 3 — `loc` includes the end | `iloc[0:2]` gives 2 |
| `astype(int)` on 1.7 | gives 1 (cut, not rounded) | `.round().astype(int)` |
| `astype(int)` on a column with NaN | error | fill first, or use `"Int64"` |
| `x == np.nan` | always False | use `.isna()` |

In [ ]:
print(np.nan == np.nan)                        # False - that's why we use isna()
x = pd.DataFrame({"a": [1, 2, 3, 4]})
print(len(x.loc[0:2]), "rows with loc[0:2] |", len(x.iloc[0:2]), "rows with iloc[0:2]")
print(pd.Series([1.7, 2.2]).astype(int).tolist(), "vs", pd.Series([1.7, 2.2]).round().astype(int).tolist())

try:
    x[x["a"] > 1 and x["a"] < 4]
except ValueError as e:
    print("ValueError:", e)
print(x[(x["a"] > 1) & (x["a"] < 4)])

## 24. Pandas for AI 🔴

**The standard ML data-preparation steps:**

```
Load → Look → Clean → Split train/test → Fill missing → Encode → Scale → NumPy arrays → Model
```

### 24.1 A small preprocessing pipeline

**Scaling** puts numbers on a similar range so big numbers (income) don't dominate small ones (age):

| Method | Formula | Result |
|---|---|---|
| Min-max | (x − min) / (max − min) | between 0 and 1 |
| Standardisation | (x − mean) / std | mean 0, std 1 |

In [ ]:
raw = pd.DataFrame({
    "age": [25, np.nan, 35, 40, 22],
    "city": ["Kabul", "Herat", "Kabul", np.nan, "Mazar"],
    "income": [50000, 60000, np.nan, 80000, 45000]
})

raw["age"] = raw["age"].fillna(raw["age"].median())          # 1. fill missing
raw["income"] = raw["income"].fillna(raw["income"].median())
raw["city"] = raw["city"].fillna("Unknown")

encoded = pd.get_dummies(raw, columns=["city"], dtype=int)    # 2. encode

for col in ["age", "income"]:                                 # 3. scale (min-max)
    encoded[col] = (encoded[col] - encoded[col].min()) / (encoded[col].max() - encoded[col].min())
print(encoded)

X = encoded.to_numpy(dtype="float32")                         # 4. NumPy for the model
print(X.shape, X.dtype)

### 24.2 Which features relate to the target?

A quick correlation check before modelling.

In [ ]:
df_features = pd.DataFrame({
    "study_hours": [1, 2, 3, 4, 5, 6, 7, 8],
    "sleep_hours": [8, 7, 7, 6, 6, 5, 5, 4],
    "score": [50, 55, 60, 65, 70, 75, 85, 90]
})
print(df_features.corr()["score"].drop("score").sort_values(key=abs, ascending=False))

`sleep_hours` is strongly **negatively** related to score here — but that doesn't mean less sleep **causes** better scores. In this data, students who study more simply sleep less.

### 24.3 Class balance

If one class is much bigger (e.g. 95% "not fraud"), a model can score 95% accuracy by always guessing the big class. **Always check the target first.**

For the next examples we need more rows, so we create a bigger student dataset (300 students). You don't need to understand every line of this cell — it just makes realistic practice data.

In [ ]:
rng = np.random.default_rng(0)
n = 300
hours = rng.uniform(0, 10, n).round(1)
attend = rng.uniform(50, 100, n).round(1)
big_students = pd.DataFrame({
    "student_id": range(1, n + 1),
    "gender": rng.choice(["F", "M"], n),
    "province": rng.choice(["Kabul", "Herat", "Balkh", "Kandahar", "Nangarhar"], n),
    "age": rng.integers(17, 26, n).astype(float),
    "study_hours": hours,
    "attendance": attend,
    "math": (35 + 4 * hours + 0.25 * attend + rng.normal(0, 8, n)).clip(0, 100).round(),
})
big_students["passed"] = (big_students["math"] + rng.normal(0, 5, n) >= 70).astype(int)
big_students.loc[rng.choice(n, 20, replace=False), "attendance"] = np.nan     # some missing values
big_students.loc[rng.choice(n, 10, replace=False), "province"] = np.nan

print(big_students.shape)
print(big_students["passed"].value_counts(normalize=True).round(2))

### 24.4 Train / test split

- **Random split:** shuffle with `sample(frac=1)`, then cut at 80%.
- **Stratified split:** keep the **same class balance** in train and test. With Pandas: take 80% **of each class** using `groupby().sample()`.

In [ ]:
shuffled = big_students.sample(frac=1, random_state=42).reset_index(drop=True)
cut = int(0.8 * len(shuffled))
train_random, test_random = shuffled[:cut], shuffled[cut:]
print("random split:", len(train_random), len(test_random))

train = big_students.groupby("passed").sample(frac=0.8, random_state=42)
test = big_students.drop(train.index)                  # everything not in train
print(pd.DataFrame({"all": big_students["passed"].value_counts(normalize=True),
                    "train": train["passed"].value_counts(normalize=True),
                    "test": test["passed"].value_counts(normalize=True)}).round(3))

### 24.5 Data leakage ⚠️

**Data leakage** = information from the **test set** sneaks into training. Example: filling missing values with the mean of the **whole** dataset. The model then looks better in testing than it really is.

**Rule:** split **first**. Then calculate every number you need (median, mean, std, list of categories) from the **training set only**, and use those same numbers on the test set.

In [ ]:
features = ["gender", "province", "age", "study_hours", "attendance"]
X_train, X_test = train[features].copy(), test[features].copy()
y_train, y_test = train["passed"], test["passed"]

# 1. Fill missing - using TRAIN numbers only
att_median = X_train["attendance"].median()
X_train["attendance"] = X_train["attendance"].fillna(att_median)
X_test["attendance"] = X_test["attendance"].fillna(att_median)
X_train["province"] = X_train["province"].fillna("Unknown")
X_test["province"] = X_test["province"].fillna("Unknown")

# 2. Encode
X_train["gender"] = X_train["gender"].map({"F": 0, "M": 1})
X_test["gender"] = X_test["gender"].map({"F": 0, "M": 1})
X_train = pd.get_dummies(X_train, columns=["province"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["province"], dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)     # same columns, same order

# 3. Scale - using TRAIN mean and std only
num_cols = ["age", "study_hours", "attendance"]
mean, std = X_train[num_cols].mean(), X_train[num_cols].std()
X_train[num_cols] = (X_train[num_cols] - mean) / std
X_test[num_cols] = (X_test[num_cols] - mean) / std

# 4. To NumPy
X_train_np, X_test_np = X_train.to_numpy("float32"), X_test.to_numpy("float32")
y_train_np, y_test_np = y_train.to_numpy(), y_test.to_numpy()

print(X_train.columns.tolist())
print("X_train:", X_train_np.shape, "| X_test:", X_test_np.shape)
print("any missing left?", np.isnan(X_train_np).any() or np.isnan(X_test_np).any())
X_train.head().round(2)

Note that `math` is **not** used as a feature — `passed` was created from `math`, so using it would be **target leakage** (the answer hidden in the question).

### 24.6 Hand-off to a model

The arrays are ready for scikit-learn (or PyTorch / TensorFlow). Pandas did all the preparation.

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression().fit(X_train_np, y_train_np)
    print(f"Test accuracy: {model.score(X_test_np, y_test_np):.2f}")
    print(pd.Series(model.coef_[0], index=X_train.columns).round(2).sort_values(key=abs, ascending=False).head())
except ImportError:
    print("Install scikit-learn to train a model: pip install scikit-learn")

### 24.7 Time-series features for forecasting

To predict tomorrow, each row needs information from the **past**: lags, moving averages of past values, and calendar features. Split **by date** — never shuffle time series, or the model sees the future.

In [ ]:
ts = sales.copy()
ts["lag_1"] = ts["sales"].shift(1)
ts["lag_7"] = ts["sales"].shift(7)
ts["avg_prev_7"] = ts["sales"].shift(1).rolling(7).mean()     # shift first: no peeking at today
ts["weekday"] = ts.index.dayofweek
ts = ts.dropna()

ts_train, ts_test = ts.loc[:"2023-03-15"], ts.loc["2023-03-16":]
print("train:", ts_train.index.min().date(), "to", ts_train.index.max().date())
print("test :", ts_test.index.min().date(), "to", ts_test.index.max().date())
ts_train.head().round(1)

## Pandas Cheat Sheet 📄

```python
# 🟢 Create & look
pd.Series(...), pd.DataFrame({...}), pd.date_range()
df.head(), df.tail(), df.sample(), df.shape, df.info(), df.describe(), df.dtypes
df.nunique(), df["c"].unique(), df["c"].value_counts(normalize=True)

# 🟢 Select
df["c"], df[["a","b"]], df.loc[rows, cols], df.iloc[i, j], df.at[r, c], df.iat[i, j]
df[(cond1) & (cond2)], df["c"].isin([...]), df["c"].between(a, b), df.query("c > @x")

# 🟢 Change
df["new"] = ..., df.loc[cond, "c"] = v, np.where(cond, a, b)
df.rename(columns={}), df.drop(columns=[]), df.set_index(), df.reset_index(drop=True)
df.sort_values(by, ascending), df.sort_index(), df.nlargest(n, "c"), df["c"].rank()

# 🟢 Clean
df.isna().sum(), df.dropna(subset=[]), df.fillna({col: val}), s.ffill(), s.bfill(), s.interpolate()
df.duplicated(), df.drop_duplicates(subset=[], keep="first")

# 🟢 Stats
df.mean(), median(), mode(), std(), min(), max(), quantile(), idxmax(), corr(), agg([...])
s.cumsum(), s.diff(), s.pct_change()

# 🟡 Types & text
df.astype(), pd.to_numeric(errors="coerce"), pd.to_datetime(), df.select_dtypes()
s.str.strip().str.title(), s.str.contains(), s.str.split(expand=True), s.str.extract()

# 🟡 Functions, outliers, encoding
s.map(dict), s.apply(f), df.apply(f, axis=1), df.map(f), np.select()
s.clip(), pd.cut(), pd.qcut(), pd.get_dummies(df, columns=[], drop_first=True, dtype=int)

# 🟡 Combine, group, reshape
pd.concat([...], ignore_index=True), pd.merge(a, b, on, how, indicator, validate), df.join()
df.groupby(k).agg(name=("col", "func")), groupby().transform(), groupby().filter(), pd.crosstab()
pd.melt(), df.pivot(), pd.pivot_table(), df.stack(), df.unstack(), df.explode()

# 🟡 Dates, plots, files
s.dt.year, s.dt.day_name(), df.resample("W").sum(), s.shift(1), s.rolling(7).mean(), s.ewm(span=7).mean()
df.plot(kind="line" | "bar" | "hist" | "box" | "scatter")
pd.read_csv(usecols, parse_dates, na_values, chunksize), df.to_csv(index=False)
pd.read_excel(), pd.read_json(), pd.json_normalize(), pd.read_parquet(), pd.read_sql()

# 🔴 Advanced / ML
df.copy(), (df.pipe(f).assign(...)), df.memory_usage(deep=True), astype("category" | "float32")
df.groupby(y).sample(frac=0.8), df.reindex(columns=train_cols, fill_value=0), df.to_numpy("float32")
```

## Checklist ✅

**🟢 Beginner**
- [ ] Series vs DataFrame, index, creating tables, `shape`, `dtypes`, `to_numpy`
- [ ] Looking at data: `head`, `tail`, `sample`, `info`, `describe`, `value_counts`, `unique`, `nunique`
- [ ] Selecting: `[]`, `loc`, `iloc`, `at`/`iat`, conditions with `& | ~`, `isin`, `between`, `query`
- [ ] Adding, renaming, dropping columns; `np.where`; `set_index`/`reset_index`
- [ ] Sorting, `nlargest`, `rank`
- [ ] Missing data: find, `dropna`, `fillna` (mean/median/mode), `ffill`/`bfill`/`interpolate`
- [ ] Duplicates
- [ ] Basic statistics, `axis`, `value_counts(normalize=True)`, `cumsum`/`diff`/`pct_change`, correlation

**🟡 Intermediate**
- [ ] Data types: `astype`, `to_numeric`, `to_datetime`, `category`, `Int64`, `select_dtypes`
- [ ] Text: `.str` methods, `split(expand=True)`, `extract`, NLP cleaning
- [ ] `map`, `apply`, `DataFrame.map`, `np.select`, vectorization vs `apply`
- [ ] Outliers (IQR, z-score, `clip`), binning (`cut`, `qcut`)
- [ ] Encoding: mapping, one-hot, `drop_first`
- [ ] `concat`, `merge` (4 join types, `indicator`, `validate`, `suffixes`), `join`
- [ ] GroupBy: `agg`, named aggregation, `transform`, `filter`, group-wise filling, `crosstab`
- [ ] Reshaping: `melt`, `pivot`, `pivot_table`, MultiIndex, `stack`/`unstack`, `explode`
- [ ] Dates: `.dt`, `Timedelta`, `resample`, `shift`, `rolling`, `expanding`, `ewm`
- [ ] Plotting from Pandas
- [ ] Files: CSV options, chunks, JSON, `json_normalize`, Excel, Parquet, SQL

**🔴 Advanced**
- [ ] Copy-on-Write, `.copy()`, no chained assignment
- [ ] Method chaining with `assign` and `pipe`
- [ ] Speed & memory: vectorize, `itertuples`, build lists then DataFrame, downcasting, `category`
- [ ] Common mistakes
- [ ] ML prep: pipeline, feature correlation, class balance, random & stratified split, **data leakage**, column alignment, NumPy hand-off, time-series features

---
# Practice Questions ✍️

Try each question in the empty cell below it **before** looking at the solutions at the end.

**Q1. 🟢** Create a DataFrame with 5 rows and at least one missing value. Run `.info()` and `.isna().sum()`, then fill the missing values with the column mean.

In [ ]:
# Write your answer here

**Q2. 🟢** Using `students_clean`, show the names of students whose `science` score is above the average science score. Use `query()` with `@`.

In [ ]:
# Write your answer here

**Q3. 🟢** Sort `students_clean` by `english` from highest to lowest and show the top 3 names and scores. Then do the same with `nlargest()`.

In [ ]:
# Write your answer here

**Q4. 🟡** Create a DataFrame with a text column of at least 3 categories. One-hot encode it with `pd.get_dummies()`, then again with `drop_first=True`. Explain the difference in a comment.

In [ ]:
# Write your answer here

**Q5. 🟡** Time it: square a 200,000-row column using `.apply()` and using a vectorized operation. Which is faster, and by about how much?

In [ ]:
# Write your answer here

**Q6. 🟡** Make a small **wide** DataFrame (one row per student, one column per subject) and turn it into **long** format with `pd.melt()`. Then turn it back with `pivot()`.

In [ ]:
# Write your answer here

**Q7. 🟡** Using `pd.date_range`, create 14 days of made-up sales data. Add a 3-day and a 7-day rolling average. In a comment, say which one reacts faster to a sudden spike.

In [ ]:
# Write your answer here

**Q8. 🟡** You have `orders = pd.DataFrame({'order_id':[1,2,3,4], 'customer_id':[10,11,10,12]})` and `customers = pd.DataFrame({'customer_id':[10,11,13], 'name':['Ali','Sara','Omar']})`. Merge them so every order is kept, and use `indicator=True` to find orders with an unknown customer.

In [ ]:
# Write your answer here

**Q9. 🔴** Build a mini preprocessing pipeline like 24.1 on a DataFrame with 2 number columns and 1 text column, with some missing values. Print the final NumPy array and its dtype.

In [ ]:
# Write your answer here

**Q10. 🔴** Using `big_students`, find which feature (`study_hours`, `attendance`, `age`) has the strongest correlation with `math`. In a comment, explain why this does **not** prove it causes higher scores.

In [ ]:
# Write your answer here

**Q11. 🔴** Split `big_students` 80/20, stratified on `passed`. Standardise `study_hours` using the **training** mean and std only. Explain in a comment why using the full-data mean would be data leakage.

In [ ]:
# Write your answer here

---
# Solutions 💡

There is often more than one correct answer.

In [ ]:
# Q1
q1 = pd.DataFrame({"x": [1.0, 2.0, np.nan, 4.0, 5.0], "y": [10, np.nan, 30, 40, 50]})
q1.info()
print(q1.isna().sum())
q1 = q1.fillna(q1.mean())
print(q1)

In [ ]:
# Q2
avg_science = students_clean["science"].mean()
print(students_clean.query("science > @avg_science")[["name", "science"]])

In [ ]:
# Q3
print(students_clean.sort_values("english", ascending=False)[["name", "english"]].head(3))
print(students_clean.nlargest(3, "english")[["name", "english"]])

In [ ]:
# Q4
colors = pd.DataFrame({"color": ["red", "green", "blue", "green"]})
print(pd.get_dummies(colors, columns=["color"], dtype=int))
print(pd.get_dummies(colors, columns=["color"], drop_first=True, dtype=int))
# drop_first removes one column ("blue"). If green=0 and red=0, it must be blue,
# so 2 columns hold the same information and avoid the dummy variable trap.

In [ ]:
# Q5
q5 = pd.DataFrame({"x": np.random.randn(200_000)})
start = time.time(); q5["a"] = q5["x"].apply(lambda v: v ** 2); t_a = time.time() - start
start = time.time(); q5["b"] = q5["x"] ** 2; t_v = time.time() - start
print(f"apply {t_a:.4f}s | vectorized {t_v:.4f}s | about {t_a / t_v:.0f}x faster")

In [ ]:
# Q6
wide6 = pd.DataFrame({"student": ["Ali", "Sara", "Omar"], "math": [80, 90, 70], "english": [75, 85, 95]})
long6 = wide6.melt(id_vars="student", var_name="subject", value_name="score")
print(long6)
print(long6.pivot(index="student", columns="subject", values="score").reset_index())

In [ ]:
# Q7
q7 = pd.DataFrame({"sales": [100, 102, 98, 101, 99, 103, 100, 250, 101, 99, 100, 102, 98, 101]},
                  index=pd.date_range("2024-01-01", periods=14))
q7["avg_3"] = q7["sales"].rolling(3).mean().round(1)
q7["avg_7"] = q7["sales"].rolling(7).mean().round(1)
print(q7)
# The 3-day average reacts faster: the spike is 1 of only 3 values,
# but 1 of 7 values in the 7-day window, so its effect is smaller there.

In [ ]:
# Q8
orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "customer_id": [10, 11, 10, 12]})
customers = pd.DataFrame({"customer_id": [10, 11, 13], "name": ["Ali", "Sara", "Omar"]})
m8 = pd.merge(orders, customers, on="customer_id", how="left", indicator=True)
print(m8)
print(m8[m8["_merge"] == "left_only"])     # order 4: customer 12 is unknown

In [ ]:
# Q9
q9 = pd.DataFrame({"height": [170, np.nan, 165, 180], "weight": [70, 80, np.nan, 90],
                   "team": ["red", "blue", np.nan, "red"]})
q9[["height", "weight"]] = q9[["height", "weight"]].fillna(q9[["height", "weight"]].median())
q9["team"] = q9["team"].fillna("Unknown")
q9 = pd.get_dummies(q9, columns=["team"], dtype=int)
for c in ["height", "weight"]:
    q9[c] = (q9[c] - q9[c].mean()) / q9[c].std()
arr9 = q9.to_numpy(dtype="float32")
print(arr9, arr9.dtype)

In [ ]:
# Q10
corr10 = big_students[["study_hours", "attendance", "age", "math"]].corr()["math"].drop("math")
print(corr10.round(3))
print("strongest:", corr10.abs().idxmax())
# Correlation only shows they move together. Another factor (e.g. motivation) could cause
# both more studying AND higher scores. Proving cause needs experiments or causal methods.

In [ ]:
# Q11
tr = big_students.groupby("passed").sample(frac=0.8, random_state=1)
te = big_students.drop(tr.index)
m11, s11 = tr["study_hours"].mean(), tr["study_hours"].std()
tr = tr.assign(study_hours_z=(tr["study_hours"] - m11) / s11)
te = te.assign(study_hours_z=(te["study_hours"] - m11) / s11)
print(len(tr), len(te), round(m11, 3), round(s11, 3))
# Using the full-data mean lets test rows influence the training data. The test set is
# then no longer truly "unseen", so the test score would look better than reality.

---
**End of notebook 🎉** You now have the full Pandas toolkit for AI: looking at data, cleaning it, transforming it, combining it and preparing it for models. Next in the AAII roadmap: **Matplotlib** for visualization.